# MAE pretrain + fine-tune на CIFAR-10

Masked Autoencoder (He et al., 2021, arXiv:2111.06377) на примитивах `spartan_torch`:

1. **Примитивы библиотеки.** `PatchEmbedding`, `TransformerBlock`, `MaskedToken`, `RandomPatchMasking`, `MAEDecoderHead`.
2. **Асимметричный autoencoder.** Encoder-ViT работает только на видимых патчах (без `[CLS]` и без mask-token'ов — отсюда ускорение ~3-4x), лёгкий decoder восстанавливает пер-патч нормализованные пиксели.
3. **Перенос.** Encoder-веса (`patch_embed` + `blocks` + `norm`) переносятся в классификационный ViT для fine-tune.

Редуцированная конфигурация (CIFAR-10, ViT-Tiny-подобный) — быстрая проверка, что пайплайн рабочий.

Запуск: `uv sync --extra dev --extra experiments`, затем `uv run jupyter lab`.

In [1]:
# === Setup: локально / devcontainer / Colab ===
import sys

IN_COLAB = "google.colab" in sys.modules
MLFLOW_ENABLED = not IN_COLAB

if IN_COLAB:
    import subprocess
    from pathlib import Path
    PROJECT_ROOT = Path("/content/spartan-torch")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/mievst/spartan-torch.git", str(PROJECT_ROOT)],
            check=True,
        )
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "--upgrade-strategy", "eager", "numpy", "scipy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e",
         f"{PROJECT_ROOT}\[experiments,dev]"],
        check=True,
    )
else:
    PROJECT_ROOT = None

print(f"IN_COLAB={IN_COLAB} | MLFLOW_ENABLED={MLFLOW_ENABLED} | python={sys.version.split()[0]}")

IN_COLAB=False | MLFLOW_ENABLED=True | python=3.13.14


<>:26: SyntaxWarning: invalid escape sequence '\['
<>:26: SyntaxWarning: invalid escape sequence '\['
C:\Users\mievst\AppData\Local\Temp\ipykernel_41716\3539442534.py:26: SyntaxWarning: invalid escape sequence '\['
  f"{PROJECT_ROOT}\[experiments,dev]"],


## 0. Константы и пути

In [2]:
from pathlib import Path

import torch

ROOT = (PROJECT_ROOT / "experiments/vit/mae") if IN_COLAB else Path.cwd()
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE}")

# --- training ---
EPOCHS = 60
BATCH_SIZE = 128
LR = 1.5e-3
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 5
NUM_WORKERS = 0
SEED = 0

# --- MAE (ViT-Tiny-подобный encoder) ---
IMG_SIZE = 32
PATCH_SIZE = 4
MASK_RATIO = 0.75
ENC_EMBED = 192
ENC_DEPTH = 6
ENC_HEADS = 3
DEC_EMBED = 96
DEC_DEPTH = 2
DEC_HEADS = 3

torch.manual_seed(SEED)
torch.set_float32_matmul_precision("medium")

MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENT_NAME = "mae-cifar10"

cwd: a:\projects\spartan-torch\experiments\vit\mae | device: cuda


## 1. Модель

In [3]:
import sys
sys.path.insert(0, str(ROOT))

from mae_model import MAEModel
from mae_lightning import MAEPretrainLightning, MAEFinetuneLightning

mae = MAEModel(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=3,
    mask_ratio=MASK_RATIO,
    encoder_embed_dim=ENC_EMBED,
    encoder_depth=ENC_DEPTH,
    encoder_heads=ENC_HEADS,
    decoder_embed_dim=DEC_EMBED,
    decoder_depth=DEC_DEPTH,
    decoder_heads=DEC_HEADS,
)
n = sum(p.numel() for p in mae.parameters() if p.requires_grad)
print(f"MAE params: {n:,} | patches={mae.num_patches} | mask={mae.mask_ratio}")

W0905 23:51:44.696000 41716 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


MAE params: 2,922,096 | patches=64 | mask=0.75


## 2. Данные

In [4]:
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_ds = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
val_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=val_transform)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

a:\projects\spartan-torch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 | Val: 10000


## 3. Pretrain

In [5]:
from collections import defaultdict
from lightning.pytorch.callbacks import Callback, ModelCheckpoint


class MetricsCollector(Callback):
    def __init__(self):
        super().__init__()
        self.metrics = defaultdict(list)

    def on_validation_epoch_end(self, trainer, pl_module):
        for k, v in trainer.logged_metrics.items():
            self.metrics[k].append(v.cpu().item())


metrics_cb = MetricsCollector()
pretrain_cb = ModelCheckpoint(
    dirpath=CKPT_DIR, filename="mae-pretrain-{epoch}",
    monitor="val/loss", mode="min", save_top_k=1, save_last=True,
)

In [ ]:
import sys
from pathlib import Path

try:
    from _mlflow import make_logger
except ImportError:  # kernel cwd is the experiment dir, not repo root
    for _base in [Path.cwd(), *Path.cwd().parents]:
        if (_base / "_mlflow.py").exists():
            sys.path.insert(0, str(_base))
            break
        if (_base / "experiments" / "_mlflow.py").exists():
            sys.path.insert(0, str(_base / "experiments"))
            break
    from _mlflow import make_logger

logger = make_logger(MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, ROOT / "mlruns", enabled=MLFLOW_ENABLED)


In [7]:
import lightning as L

pre = MAEPretrainLightning(
    mae, lr=LR, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS,
)

pretrainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[metrics_cb, pretrain_cb],
    logger=logger,
    precision="16-mixed",
    benchmark=True,
)
pretrainer.fit(pre, train_loader, val_loader)

if logger is not None:
    print(f"MLflow run: {MLFLOW_TRACKING_URI}/#/experiments/{logger.experiment_id}/runs/{logger.run_id}")
print("Best pretrain ckpt:", pretrain_cb.best_model_path)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Experiment with name mae-cifar10 not found. Creating it.


RestException: RESOURCE_ALREADY_EXISTS: Experiment(name=mae-cifar10) already exists. Error: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "uq_experiments_workspace_name"
DETAIL:  Key (workspace, name)=(default, mae-cifar10) already exists.

[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (%(name)s, %(workspace)s, %(artifact_location)s, %(lifecycle_stage)s, %(creation_time)s, %(last_update_time)s) RETURNING experiments.experiment_id]
[parameters: {'name': 'mae-cifar10', 'workspace': 'default', 'artifact_location': '', 'lifecycle_stage': 'active', 'creation_time': 1788641512924, 'last_update_time': 1788641512924}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

## 4. Визуализация реконструкции

In [ ]:
import matplotlib.pyplot as plt
import torch


@torch.no_grad()
def show_reconstruction(model, image, title=""):
    model.eval()
    out = model(image.unsqueeze(0))
    mask = out["mask"][0]                     # L
    pred = out["pred"][0]                     # L, pixels (normalized)
    target = out["target"][0]
    C, H, W = image.shape
    p = model.patch_size
    L = model.num_patches

    def unpatch(tokens, show_mask=False):
        tmp = tokens.view(L, C, p, p) if not show_mask else tokens
        grid = tmp.permute(0, 2, 3, 1) if not show_mask else None
        if show_mask:
            return None
        return grid.reshape(H, W, C).cpu().numpy()

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    img = image.permute(1, 2, 0).cpu().numpy()
    ax[0].imshow(img); ax[0].set_title("original"); ax[0].axis("off")

    mask_img = mask.view(int(round(L ** 0.5)), int(round(L ** 0.5))).float().cpu().numpy()
    ax[1].imshow(mask_img, cmap="gray"); ax[1].set_title("mask"); ax[1].axis("off")

    recon = unpatch(pred)
    ax[2].imshow(recon); ax[2].set_title("reconstruction"); ax[2].axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


sample = next(iter(val_loader))[0][0]
show_reconstruction(mae, sample)

## 5. Fine-tune

In [ ]:
# переносим encoder-веса из best pretrain-чекпоинта в классификационный ViT
best_pretrain = pretrain_cb.best_model_path

ft = MAEFinetuneLightning(
    img_size=IMG_SIZE, patch_size=PATCH_SIZE, in_channels=3, num_classes=10,
    embed_dim=ENC_EMBED, depth=ENC_DEPTH, num_heads=ENC_HEADS,
    pretrain_ckpt=best_pretrain,
    lr=1e-3, weight_decay=WEIGHT_DECAY, warmup_epochs=5, max_epochs=40,
)

In [ ]:
from collections import defaultdict
from lightning.pytorch.callbacks import Callback, ModelCheckpoint, EarlyStopping


class ClsCollector(Callback):
    def __init__(self):
        super().__init__()
        self.metrics = defaultdict(list)

    def on_validation_epoch_end(self, trainer, pl_module):
        for k, v in trainer.logged_metrics.items():
            self.metrics[k].append(v.cpu().item())


cls_cb = ClsCollector()
ft_cp = ModelCheckpoint(
    dirpath=CKPT_DIR, filename="mae-finetune-{epoch}",
    monitor="val/cls_acc", mode="max", save_top_k=1, save_last=True,
)
early = EarlyStopping(monitor="val/cls_acc", mode="max", patience=10, verbose=True)

In [ ]:
ft_trainer = L.Trainer(
    max_epochs=40,
    accelerator="auto",
    callbacks=[cls_cb, ft_cp, early],
    logger=logger,
    precision="16-mixed",
    benchmark=True,
)
ft_trainer.fit(ft, train_loader, val_loader)
print("Best finetune ckpt:", ft_cp.best_model_path)

## 6. Графики

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

m = metrics_cb.metrics
pre_epochs = range(1, len(m["train/loss"]) + 1)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(pre_epochs, m["train/loss"], label="train (pretrain)")
ax.plot(pre_epochs, m["val/loss"], label="val (pretrain)")
ax.set_title("MAE reconstruction loss")
ax.set_xlabel("epoch")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
m = cls_cb.metrics
ft_epochs = range(1, len(m["val/cls_acc"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ft_epochs, m["train/cls_loss"], label="train")
ax1.plot(ft_epochs, m["val/cls_loss"], label="val")
ax1.set_title("Fine-tune loss"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ft_epochs, m["val/cls_acc"], label="val_acc")
ax2.set_title("Fine-tune accuracy"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()